# U-Net Segmentation Training - Google Colab Orchestrator

**Orchestrator only.** All training logic lives in the main repository files
(`train.py`, `utils/data_loading.py`, `unet/`, etc.). This notebook simply clones
the repo, installs dependencies, and calls into the existing code.

---

| Item | Value |
|------|-------|
| Source repo | `https://github.com/HaikalFK/segmentasi-unet.git` |
| Dataset | Plant Phenotyping (Kaggle) |
| Classes | 22 (auto-detected) |
| Runtime | **GPU** (T4, V100, or A100) |

---
## 1. Install Dependencies

PyTorch with CUDA is pre-installed in Colab. We only need to install
the project-specific dependencies from `requirements.txt`.

In [2]:
# Verify GPU is available
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device: {torch.cuda.get_device_name(0)}')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU device: Tesla T4


In [ ]:
# Clone the repository (or pull latest if already cloned)
import os
from pathlib import Path

REPO_URL = 'https://github.com/HaikalFK/segmentasi-unet.git'
REPO_DIR = Path('/content/segmentasi-unet')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Fetching latest...')
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/train_config

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Clone the repository (or pull latest if already cloned)
import os
from pathlib import Path

REPO_URL = 'https://github.com/HaikalFK/segmentasi-unet.git'
REPO_DIR = Path('/content/segmentasi-unet')

if not REPO_DIR.exists():
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already cloned. Fetching latest...')
    %cd {REPO_DIR}
    !git fetch --all
    !git reset --hard origin/train_config

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

# Install project dependencies (version ranges to avoid build failures on Colab)
# PyTorch with CUDA is already pre-installed in Colab.
!pip install --quiet --upgrade pip
!pip install --quiet -r requirements.txt

# Verify imports
from utils.data_loading import BasicDataset
from unet import UNet
print('All imports OK')

In [5]:
!python data/download_dataset.py

Plant Phenotyping Dataset Downloader

Downloading... (this may take a while for large datasets)
100% 1.11G/1.11G [00:56<00:00, 21.0MB/s]
Extracting files...
Dataset downloaded to: /root/.cache/kagglehub/datasets/pillisiddharth/plant-phenotyping-dataset/versions/1
Dataset root: /root/.cache/kagglehub/datasets/pillisiddharth/plant-phenotyping-dataset/versions/1/Plant_Phenotyping_Datasets

Organizing dataset into data/imgs/ and data/masks/

  Processing: Plant/Ara2012
    Found 120 RGB images
    Copied: 120 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Ara2013-Canon
    Found 165 RGB images
    Copied: 165 image/mask pairs, Skipped: 0 (no label)

  Processing: Plant/Tobacco
    Found 62 RGB images
    Copied: 62 image/mask pairs, Skipped: 0 (no label)

  Total: 347 image/mask pairs copied

Verifying dataset
  Images: 347
  Masks:  347
  Matched pairs: 347

Done! Dataset is ready for training.
  Images: /content/segmentasi-unet/data/imgs  (347 files)
  Masks:  /content/segm

---
## 3. Run Training

All parameters are passed as CLI arguments to `train.py`.
Edit the variables below to configure training.

In [6]:
# ============================================================
# TRAINING CONFIGURATION
# Edit these values as needed.
# ============================================================
EPOCHS        = 100
BATCH_SIZE    = 8
SCALE         = 0.5
LEARNING_RATE = 1e-5
VALIDATION    = 10.0
AMP           = True
BILINEAR      = False

print('Configuration:')
print(f'  Epochs:    {EPOCHS}')
print(f'  Batch:     {BATCH_SIZE}')
print(f'  Scale:     {SCALE}')
print(f'  AMP:       {AMP}')
print(f'  Bilinear:  {BILINEAR}')

Configuration:
  Epochs:    100
  Batch:     8
  Scale:     0.5
  AMP:       True
  Bilinear:  False


---
## ⚠️ TRAINING CELL DUPLIKAT — DIHAPUS

Gunakan cell **"4. Run Training (with Early Stopping)"** di bawah yang sudah pakai:
- ✅ Early stopping (patience=15, delta=0.001)
- ✅ AMP (mixed precision)
- ✅ Auto-detect classes dari dataset

Training di cell ini (bawaan lama) sudah tidak dipakai lagi. Silakan scroll ke bawah.

# ============================================================
# TRAINING CONFIGURATION
# Edit these values as needed. All arguments are passed
# directly to train.py from the main repo.
# ============================================================

In [7]:
EPOCHS              = 100
BATCH_SIZE          = 8
SCALE               = 0.5
LEARNING_RATE       = 1e-5
VALIDATION          = 10.0
AMP                 = True
BILINEAR            = False
EARLY_STOP_PATIENCE = 15
EARLY_STOP_DELTA    = 0.001

print('Configuration:')
print(f'  Epochs:              {EPOCHS}')
print(f'  Batch:               {BATCH_SIZE}')
print(f'  Scale:               {SCALE}')
print(f'  AMP:                 {AMP}')
print(f'  Bilinear:            {BILINEAR}')
print(f'  Early stop patience: {EARLY_STOP_PATIENCE}')
print(f'  Early stop delta:    {EARLY_STOP_DELTA}')

Configuration:
  Epochs:              100
  Batch:               8
  Scale:               0.5
  AMP:                 True
  Bilinear:            False
  Early stop patience: 15
  Early stop delta:    0.001


In [8]:
from pathlib import Path
checkpoints = sorted(Path('checkpoints').glob('*.pth'))
print(f'Checkpoints found: {len(checkpoints)}')
for ckpt in checkpoints:
    size_mb = ckpt.stat().st_size / (1024 * 1024)
    print(f'  {ckpt.name}  ({size_mb:.2f} MB)')

Checkpoints found: 0


In [9]:
# Build and execute the training command
cmd = (
    f'python train.py'
    f' --classes 21'
    f' --epochs {EPOCHS}'
    f' --batch-size {BATCH_SIZE}'
    f' --scale {SCALE}'
    f' --learning-rate {LEARNING_RATE}'
    f' --validation {VALIDATION}'
    f' --early-stop-patience {EARLY_STOP_PATIENCE}'
    f' --early-stop-delta {EARLY_STOP_DELTA}'
)

if AMP:
    cmd += ' --amp'
if BILINEAR:
    cmd += ' --bilinear'

print(f'Command: {cmd}')
print('=' * 70)
!{cmd}

Command: python train.py --classes 21 --epochs 100 --batch-size 8 --scale 0.5 --learning-rate 1e-05 --validation 10.0 --early-stop-patience 15 --early-stop-delta 0.001 --amp
INFO: Using device cuda
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:07<00:00, 46.13it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
INFO: Detected 22 classes from mask files (override --classes 21)
INFO: Network:
	3 input channels
	22 output channels (classes)
	Transposed conv upscaling
INFO: Creating dataset with 347 examples
INFO: Scanning mask files to determine unique values
100% 347/347 [00:06<00:00, 56.54it/s]
INFO: Unique mask values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 27]
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
]11;?]11;?wandb: WARNING `resume` will be ignored since W&B syncing is set t

---
## 5. Evaluate Model (Custom - Full Metrics)

Run comprehensive evaluation: Dice Score, mIoU, Per-class IoU, and formatted table.

In [10]:
# ============================================================
# CUSTOM EVALUATION - Full metrics in notebook (no base code changes needed)
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from pathlib import Path
from tabulate import tabulate

from utils.data_loading import BasicDataset, CarvanaDataset
from utils.dice_score import multiclass_dice_coeff, dice_coeff
from unet import UNet
from torch.utils.data import DataLoader, random_split

CHECKPOINT_PATH = 'checkpoints/checkpoint_best.pth'
SCALE = 0.5
BATCH_SIZE = 8
VALIDATION = 10.0
AMP = True
BILINEAR = False

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
mask_values = state_dict.pop('mask_values', [0, 1])
n_classes = len(mask_values)

print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Classes: {n_classes} (mask_values: {mask_values})')
print(f'Device: {device}')

# Create model
net = UNet(n_channels=3, n_classes=n_classes, bilinear=BILINEAR)
net.to(device=device)
net.load_state_dict(state_dict)
net.eval()

# Create validation dataset
try:
    dataset = CarvanaDataset('./data/imgs/', './data/masks/', SCALE, target_size=(256, 256))
except (AssertionError, RuntimeError, IndexError):
    dataset = BasicDataset('./data/imgs/', './data/masks/', SCALE, target_size=(256, 256))
dataset.mask_values = mask_values

n_val = int(len(dataset) * VALIDATION / 100)
n_train = len(dataset) - n_val
_, val_set = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))

val_loader = DataLoader(val_set, shuffle=False, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True, drop_last=True)
print(f'Validation samples: {len(val_set)}')
print('-' * 60)

# Evaluation
dice_score = 0.0
total_iou_per_class = None
total_mean_iou = 0.0
num_batches = 0

with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=AMP):
    for batch in tqdm(val_loader, desc='Evaluating', unit='batch'):
        image, mask_true = batch['image'], batch['mask']
        image = image.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
        mask_true = mask_true.to(device=device, dtype=torch.long)

        with torch.no_grad():
            mask_pred = net(image)

        if n_classes == 1:
            mask_pred = (F.sigmoid(mask_pred) > 0.5).float()
            dice_score += dice_coeff(mask_pred, mask_true.float(), reduce_batch_first=False).item()
        else:
            mask_true_oh = F.one_hot(mask_true, n_classes).permute(0, 3, 1, 2).float()
            mask_pred_oh = F.one_hot(mask_pred.argmax(dim=1), n_classes).permute(0, 3, 1, 2).float()
            
            # Dice (excl background)
            dice_score += multiclass_dice_coeff(mask_pred_oh[:, 1:], mask_true_oh[:, 1:], reduce_batch_first=False).item()

            # IoU per class (excl background)
            pred_masks = mask_pred.argmax(dim=1)
            true_masks = mask_true

            # Convert to one-hot for IoU
            pred_oh = F.one_hot(pred_masks, n_classes).permute(0, 3, 1, 2).float()
            true_oh = F.one_hot(true_masks, n_classes).permute(0, 3, 1, 2).float()

            # Exclude background (class 0)
            pred_oh = pred_oh[:, 1:]
            true_oh = true_oh[:, 1:]

            intersection = (pred_oh * true_oh).sum(dim=(0, 2, 3))
            union = pred_oh.sum(dim=(0, 2, 3)) + true_oh.sum(dim=(0, 2, 3)) - intersection
            iou = intersection / (union + 1e-8)
            mean_iou = iou.mean()

            if total_iou_per_class is None:
                total_iou_per_class = iou.detach().cpu()
            else:
                total_iou_per_class += iou.detach().cpu()
            total_mean_iou += mean_iou.item()

        num_batches += 1

avg_dice = dice_score / num_batches
avg_iou_per_class = total_iou_per_class / num_batches if total_iou_per_class is not None else None
avg_mean_iou = total_mean_iou / num_batches if num_batches > 0 else 0.0

# ============================================================
# PRINT RESULTS
# ============================================================
print('\n' + '=' * 70)
print('EVALUATION RESULTS')
print('=' * 70)
print(f'Dice Score (excl. background): {avg_dice:.4f}')
print(f'Mean IoU (excl. background):   {avg_mean_iou:.4f}')
print('-' * 70)
if avg_iou_per_class is not None:
    print('IoU per class (excl. background):')
    table_data = []
    for i, iou in enumerate(avg_iou_per_class):
        class_idx = i + 1  # skip background
        class_val = mask_values[class_idx] if class_idx < len(mask_values) else class_idx
        table_data.append([f'Class {class_val}', f'{iou:.4f}'])
    print(tabulate(table_data, headers=['Class', 'IoU'], tablefmt='grid'))
print('=' * 70)

# Also return as dict for potential further use
results = {
    'dice': avg_dice,
    'mean_iou': avg_mean_iou,
    'iou_per_class': avg_iou_per_class.tolist() if avg_iou_per_class is not None else None,
}
results

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/checkpoint_best.pth'

---
## 6. Predict on Sample Image (Custom - Full Visualization)

Predict with IoU computation vs ground truth, save to `outputs/`, side-by-side visualization.

In [ ]:
# ============================================================
# CUSTOM PREDICTION - Full visualization with IoU
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
from pathlib import Path
from tabulate import tabulate

from utils.data_loading import BasicDataset
from unet import UNet

# Config
CHECKPOINT_PATH = 'checkpoints/checkpoint_best.pth'
SAMPLE_IMAGE = 'data/imgs/ara2012_plant001.png'
SCALE = 0.5
BILINEAR = False
OUTPUT_DIR = 'outputs'

# Load checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
mask_values = state_dict.pop('mask_values', [0, 1])
n_classes = len(mask_values)

print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Classes: {n_classes}')
print(f'Mask values: {mask_values}')
print(f'Device: {device}')

# Create model
net = UNet(n_channels=3, n_classes=n_classes, bilinear=BILINEAR)
net.to(device=device)
net.load_state_dict(state_dict)
net.eval()

# Load image
img = Image.open(SAMPLE_IMAGE).convert('RGB')
print(f'\nInput: {SAMPLE_IMAGE} ({img.size[0]}x{img.size[1]})')

# Preprocess & Predict
img_tensor = torch.from_numpy(BasicDataset.preprocess(None, img, SCALE, is_mask=False))
img_tensor = img_tensor.unsqueeze(0).to(device=device, dtype=torch.float32)

with torch.no_grad():
    output = net(img_tensor)
    output = F.interpolate(output, (img.size[1], img.size[0]), mode='bilinear', align_corners=False)
    if n_classes > 1:
        pred_mask = output.argmax(dim=1)[0].cpu().numpy()
    else:
        pred_mask = (F.sigmoid(output) > 0.5).float()[0, 0].cpu().numpy()

# Save prediction
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_name = Path(SAMPLE_IMAGE).stem + '_OUT.png'
out_path = Path(OUTPUT_DIR) / out_name

def mask_to_image(mask, mask_values):
    out = np.zeros((mask.shape[-2], mask.shape[-1]), dtype=np.uint8)
    for i, v in enumerate(mask_values):
        out[mask == i] = v
    return Image.fromarray(out)

result_img = mask_to_image(pred_mask, mask_values)
result_img.save(out_path)
print(f'Prediction saved to: {out_path}')

# ============================================================
# COMPUTE IoU vs GROUND TRUTH (if available)
# ============================================================
mask_dir = 'data/masks/'
base_name = Path(SAMPLE_IMAGE).stem
mask_path = Path(mask_dir) / (base_name + '.png')

iou_per_class = None
mean_iou = None

if mask_path.exists():
    true_mask = np.array(Image.open(mask_path))
    
    # Compute IoU
    pred_oh = F.one_hot(torch.from_numpy(pred_mask), n_classes).permute(2, 0, 1).float()
    true_oh = F.one_hot(torch.from_numpy(true_mask), n_classes).permute(2, 0, 1).float()
    
    # Exclude background (class 0)
    pred_oh = pred_oh[1:]
    true_oh = true_oh[1:]
    
    intersection = (pred_oh * true_oh).sum(dim=(1, 2))
    union = pred_oh.sum(dim=(1, 2)) + true_oh.sum(dim=(1, 2)) - intersection
    iou = intersection / (union + 1e-8)
    iou_per_class = iou.numpy()
    mean_iou = iou.mean().item()
    
    print(f'\nGround truth: {mask_path}')
    print(f'Mean IoU (excl. background): {mean_iou:.4f}')
    print('-' * 50)
    print('IoU per class:')
    table_data = []
    for i, iou_val in enumerate(iou_per_class):
        class_idx = i + 1
        class_val = mask_values[class_idx] if class_idx < len(mask_values) else class_idx
        table_data.append([f'Class {class_val}', f'{iou_val:.4f}'])
    print(tabulate(table_data, headers=['Class', 'IoU'], tablefmt='grid'))
else:
    print(f'\nNo ground truth found at {mask_path} - skipping IoU computation')

# ============================================================
# VISUALIZATION
# ============================================================
# Create color palette for visualization
def create_color_palette(n_classes):
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(n_classes, 3), dtype=np.uint8)
    colors[0] = [0, 0, 0]  # background = black
    return colors

colors = create_color_palette(n_classes)

def mask_to_rgb(mask, colors):
    rgb = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)
    for i in range(len(colors)):
        rgb[mask == i] = colors[i]
    return rgb

pred_rgb = mask_to_rgb(pred_mask, colors)

# Load ground truth if exists for visualization
if mask_path.exists():
    true_rgb = mask_to_rgb(true_mask, colors)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(true_rgb)
    axes[1].set_title(f'Ground Truth', fontsize=14)
    axes[1].axis('off')
    
    axes[2].imshow(pred_rgb)
    axes[2].set_title(f'Prediction (mIoU={mean_iou:.4f})', fontsize=14)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img)
    axes[0].set_title('Original Image', fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(pred_rgb)
    axes[1].set_title('Prediction', fontsize=14)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

print(f'\nDone! Output saved to {OUTPUT_DIR}/')